# Binary sentiment specialists
Run all cells in a GPU Colab session. This trains six models for a paired comparison. Both arms use the original grouped split, seeds, heads, optimizer and epoch budget. The baseline uses its validation-selected voting rule. Each binary specialist learns no/yes and the largest yes probability determines the ensemble label (ties follow label order).

In [ ]:
%%bash
set -e
cd /content

if [ -d Intro_to_AI_HW/.git ]; then
    git -C Intro_to_AI_HW pull --ff-only origin main
else
    git clone --branch main https://github.com/billlllllllly/Intro_to_AI_HW.git
fi

cd Intro_to_AI_HW/IAI_HW3
pip install -r requirements.txt

In [ ]:
# Baseline source snapshot from the workspace.
BASELINE_SOURCE = '"""Train a three-class sentiment classifier on the HW3 labeled dataset."""\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport html\nfrom pathlib import Path\nimport re\nimport subprocess\nfrom tempfile import NamedTemporaryFile\n\nimport emoji\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import classification_report, confusion_matrix, f1_score\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom torch import nn\nfrom torch.utils.data import DataLoader, WeightedRandomSampler\nfrom transformers import (\n    AutoTokenizer,\n    BertModel,\n    get_cosine_schedule_with_warmup,\n    get_linear_schedule_with_warmup,\n)\n\n\nROOT = Path(__file__).resolve().parent\nBASE_MODEL = "google-bert/bert-base-uncased"\nCOMPANY_TOKEN = "[CPY]"\nGMAIL_TOKEN = "[GMAIL]"\nLABELS = ("negative", "neutral", "positive")\nLABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}\nENCODER_LR = 1e-5\nHEAD_LR = 5e-4\n\n\ndef preprocess(text, clean=False, url_token=False, distinguish_mentions=False):\n    if clean:\n        text = html.unescape(text)\n        text = re.sub(r"Cancelled Flight(?:led|ed)", "cancelled", text, flags=re.I)\n        text = re.sub(r"Late Flightr\\b", "later", text, flags=re.I)\n    text = re.sub(r"(?:https?://|www\\.)\\S+", " [URL] " if url_token else " ", text, flags=re.IGNORECASE)\n    text = re.sub(r"[\\w.+%\\-]+@gmail\\.com(?![\\w-]|\\.[\\w-])",\n                  f" {GMAIL_TOKEN} ", text, flags=re.IGNORECASE)\n    mentions = r"(?<!\\w)@\\w+"\n    first = re.search(mentions, text) if distinguish_mentions else None\n    if first:\n        addressed = first.group().casefold()\n        text = re.sub(mentions, lambda m: " [CPY] " if m.group().casefold() == addressed\n                      else " [OTHER_USER] ", text)\n    else:\n        text = re.sub(mentions, f" {COMPANY_TOKEN} ", text)\n    text = emoji.replace_emoji(\n        text, replace=lambda symbol, data: " " + data["en"].strip(":").replace("_", " ") + " ",\n    )\n    return " ".join(text.split())\n\n\ndef has_content(text):\n    cleaned = preprocess(text).replace(COMPANY_TOKEN, "").replace(GMAIL_TOKEN, "")\n    return any(character.isalnum() for character in cleaned)\n\n\nclass CNNHead(nn.Module):\n    def __init__(self, hidden_size):\n        super().__init__()\n        self.widths = (2, 3, 4)\n        channels = 64\n        self.convolutions = nn.ModuleList(\n            nn.Conv1d(hidden_size, channels, width) for width in self.widths\n        )\n        self.dropout = nn.Dropout(0.3)\n        self.classifier = nn.Linear(channels * len(self.widths), len(LABELS))\n\n    def forward(self, hidden, attention_mask):\n        padding = max(0, max(self.widths) - hidden.size(1))\n        tokens = nn.functional.pad(hidden.transpose(1, 2), (0, padding))\n        mask = nn.functional.pad(attention_mask.bool(), (0, padding), value=False)\n        features = []\n        for width, convolution in zip(self.widths, self.convolutions):\n            activations = convolution(tokens).relu()\n            valid = mask.unfold(1, width, 1).all(dim=-1).unsqueeze(1)\n            pooled = activations.masked_fill(~valid, 0).amax(dim=-1)\n            features.append(pooled)\n        return self.classifier(self.dropout(torch.cat(features, dim=1)))\n\n\nclass SentimentModel(nn.Module):\n    def __init__(self, vocab_size, head_type="mlp"):\n        super().__init__()\n        self.encoder = BertModel.from_pretrained(BASE_MODEL, add_pooling_layer=False)\n        self.encoder.resize_token_embeddings(vocab_size)\n        if head_type == "cnn":\n            self.head = CNNHead(self.encoder.config.hidden_size)\n            return\n        self.head = nn.Sequential(\n            nn.Linear(self.encoder.config.hidden_size, 128),\n            nn.ReLU(),\n            nn.Dropout(0.3),\n            nn.Linear(128, 128),\n            nn.ReLU(),\n            nn.Dropout(0.3),\n            nn.Linear(128, len(LABELS)),\n        )\n\n    def forward(self, inputs, embeddings=None):\n        if embeddings is not None:\n            inputs = {name: tensor for name, tensor in inputs.items() if name != "input_ids"}\n            inputs["inputs_embeds"] = embeddings\n        hidden = self.encoder(**inputs).last_hidden_state\n        if isinstance(self.head, CNNHead):\n            return self.head(hidden, inputs["attention_mask"])\n        return self.head(hidden[:, 0])\n\n\ndef make_loader(frame, tokenizer, batch_size, max_length, shuffle=False, sampler=None, clean=False, url_token=False, distinguish_mentions=False):\n    labels = frame["label"].map(LABEL_TO_ID) if "label" in frame else [-1] * len(frame)\n    examples = list(zip(frame["text"], labels))\n\n    def collate(batch):\n        texts, labels = zip(*batch)\n        inputs = tokenizer(\n            [preprocess(text, clean, url_token, distinguish_mentions) for text in texts],\n            padding=True,\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        )\n        return inputs, torch.tensor(labels, dtype=torch.long)\n\n    return DataLoader(examples, batch_size=batch_size, shuffle=shuffle and sampler is None,\n                      sampler=sampler, collate_fn=collate)\n\n\ndef run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None,\n              class_weights=None, fgsm_epsilon=0.0):\n    training = optimizer is not None\n    model.train(training)\n    loss_sum = 0.0\n    predictions = []\n    targets = []\n    with torch.set_grad_enabled(training):\n        for inputs, labels in loader:\n            inputs = {name: tensor.to(device) for name, tensor in inputs.items()}\n            labels = labels.to(device)\n            if training:\n                optimizer.zero_grad()\n            embeddings = None\n            if training and fgsm_epsilon > 0:\n                embeddings = model.encoder.get_input_embeddings()(inputs["input_ids"])\n            logits = model(inputs) if embeddings is None else model(inputs, embeddings)\n            loss = loss_fn(logits, labels)\n            if training:\n                objective = loss\n                if class_weights is not None:\n                    objective = nn.functional.cross_entropy(logits, labels, weight=class_weights)\n                if embeddings is not None:\n                    gradient, = torch.autograd.grad(objective, embeddings, retain_graph=True)\n                    perturbation = fgsm_epsilon * gradient.sign()\n                    perturbation = perturbation * inputs["attention_mask"].unsqueeze(-1)\n                    adversarial_logits = model(inputs, embeddings + perturbation)\n                    adversarial_loss = nn.functional.cross_entropy(\n                        adversarial_logits, labels, weight=class_weights,\n                    )\n                    objective = (objective + adversarial_loss) / 2\n                objective.backward()\n                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)\n                optimizer.step()\n                if scheduler is not None:\n                    scheduler.step()\n\n            loss_sum += loss.item() * labels.size(0)\n            predictions.extend(logits.detach().argmax(dim=1).tolist())\n            targets.extend(labels.tolist())\n\n    macro_f1 = f1_score(\n        targets, predictions, labels=list(range(len(LABELS))),\n        average="macro", zero_division=0,\n    )\n    return loss_sum / len(targets), macro_f1\n\n\n@torch.no_grad()\ndef predict_probabilities(model, loader, device):\n    model.eval()\n    batches = []\n    for inputs, _ in loader:\n        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}\n        batches.append(model(inputs).softmax(dim=1).cpu())\n    return torch.cat(batches)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--checkpoint-dir", type=Path, help="Reuse completed models from this directory.")\n    parser.add_argument("--distinguish-mentions", action="store_true")\n    parser.add_argument("--review-path", type=Path, help="Save validation neutral/negative disagreements here.")\n    parser.add_argument("--epochs", type=int, default=5, help="Maximum number of epochs.")\n    parser.add_argument("--patience", type=int, default=2, help="Stop after this many epochs without a new best validation macro-F1.")\n    parser.add_argument("--class-weighting", choices=("sqrt", "inverse", "none"), default="sqrt",\n                        help="Training loss weighting based on training split class frequencies.")\n    parser.add_argument("--fgsm-epsilon", type=float, default=0.0,\n                        help="Embedding perturbation strength for optional FGSM training.")\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--balanced-sampling", action="store_true",\n                        help="Sample training examples inversely to class frequency.")\n    parser.add_argument("--ensemble", action="store_true",\n                        help="Train one CNN specialist and two MLP specialists.")\n    parser.add_argument("--schedule", choices=("linear", "cosine"), default="linear")\n    parser.add_argument("--max-length", type=int, default=128)\n    return run(**vars(parser.parse_args()))\n\n\ndef run(*, epochs=5, patience=2, class_weighting="sqrt", fgsm_epsilon=0.0,\n        batch_size=16, balanced_sampling=False, ensemble=False, max_length=128,\n        checkpoint_dir=None, schedule="linear", distinguish_mentions=False, review_path=None,\n        sweep_path=None, tune_neutral=False):\n    """Train BERT and return benchmark predictions and training histories."""\n    if schedule not in ("linear", "cosine"):\n        raise ValueError("Schedule must be linear or cosine.")\n    if checkpoint_dir is not None:\n        checkpoint_dir = Path(checkpoint_dir)\n    if not 0 <= fgsm_epsilon < float("inf"):\n        raise ValueError("FGSM epsilon must be finite and nonnegative.")\n    if patience < 1:\n        raise ValueError("Patience must be positive.")\n    if epochs < 1 or batch_size < 1 or not 3 <= max_length <= 512:\n        raise ValueError("Use positive epochs and batch size, and a max length from 3 to 512.")\n    if class_weighting not in ("sqrt", "inverse", "none"):\n        raise ValueError("Class weighting must be sqrt, inverse, or none.")\n\n    seed = 42\n    torch.manual_seed(seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    data = pd.read_csv(ROOT / "dataset" / "dataset.csv", keep_default_na=False)\n    if set(data["label"]) != set(LABELS):\n        raise ValueError(f"Expected exactly these sentiment labels: {LABELS}")\n\n    groups = data["text"].map(lambda text: preprocess(text).casefold())\n    outer_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n    train_ids, held_out_ids = next(outer_split.split(data, data["label"], groups))\n    train, held_out = data.iloc[train_ids], data.iloc[held_out_ids]\n    inner_split = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed)\n    val_ids, test_ids = next(\n        inner_split.split(held_out, held_out["label"], groups.iloc[held_out_ids])\n    )\n    validation, test = held_out.iloc[val_ids], held_out.iloc[test_ids]\n    keep = train["text"].map(has_content)\n    removed = train.loc[~keep, "label"].value_counts().reindex(LABELS, fill_value=0)\n    train = train.loc[keep]\n    if set(train["label"]) != set(LABELS):\n        raise ValueError("Filtering nearly empty posts left a training class without examples.")\n    print(f"Removed nearly empty training posts by class: {removed.to_dict()}")\n    print(f"Device: {device} | train: {len(train)} | validation: {len(validation)} | test: {len(test)}", flush=True)\n\n    benchmark = pd.read_csv(ROOT / "dataset" / "benchmark.csv", keep_default_na=False)\n    targets = validation["label"].map(LABEL_TO_ID).tolist()\n    counts = train["label"].value_counts().reindex(LABELS)\n    if checkpoint_dir is not None:\n        checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    # The previous Colab version hashed the entire script into its filenames.\n    # Its training recipe is compatible only with the original mention handling.\n    sources = [Path(__file__).read_bytes()]\n    if checkpoint_dir and any(checkpoint_dir.glob("*.pt")):\n        compatible = ["57eaa643d4bc977cf8ffee1496c1acb8e0fbe183"]\n        if not distinguish_mentions:\n            compatible.append("236a5a13c653087e41a9e8a79f8fad60aa780546")\n        for blob in compatible:\n            try:\n                old = subprocess.run(["git", "cat-file", "blob", blob], cwd=ROOT,\n                                     capture_output=True, check=True).stdout\n                sources.append(old)\n            except (OSError, subprocess.CalledProcessError):\n                continue\n    data_bytes = (ROOT / "dataset" / "dataset.csv").read_bytes()\n    run_key = repr((epochs, patience, batch_size, max_length, str(device),\n                    train.index.tolist(), validation.index.tolist())).encode()\n    fingerprints = []\n    for source in sources:\n        digest = hashlib.sha256(source)\n        digest.update(data_bytes)\n        digest.update(run_key)\n        fingerprints.append(digest)\n\n    def fit(model_seed, specialist, head_type):\n        key = (class_weighting, balanced_sampling, fgsm_epsilon, model_seed, specialist, head_type, schedule)\n        if distinguish_mentions:\n            key += ("distinct_mentions",)\n        checkpoint = None\n        if checkpoint_dir is not None:\n            paths = []\n            for fingerprint in fingerprints:\n                digest = fingerprint.copy()\n                digest.update(repr(key).encode())\n                paths.append(checkpoint_dir / (digest.hexdigest() + ".pt"))\n            checkpoint = next((path for path in paths if path.is_file()), paths[0])\n        torch.manual_seed(model_seed)\n        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)\n        tokens = [COMPANY_TOKEN, GMAIL_TOKEN] + (["[OTHER_USER]"] if distinguish_mentions else [])\n        tokenizer.add_special_tokens({"additional_special_tokens": tokens})\n        options = dict(tokenizer=tokenizer, batch_size=batch_size, max_length=max_length,\n                       distinguish_mentions=distinguish_mentions)\n        val_loader = make_loader(validation, **options)\n        if checkpoint is not None and checkpoint.is_file():\n            print(f"Reusing {head_type.upper()} checkpoint | seed {model_seed} | specialist {specialist}", flush=True)\n            saved = torch.load(checkpoint, map_location="cpu", weights_only=True)\n            model = SentimentModel(len(tokenizer), head_type=head_type)\n            model.load_state_dict(saved["model"])\n            model.to(device)\n            return (saved["validation"],\n                    predict_probabilities(model, make_loader(test, **options), device),\n                    predict_probabilities(model, make_loader(benchmark, **options), device),\n                    saved["history"])\n\n        sampler = None\n        if balanced_sampling:\n            sampler = WeightedRandomSampler(train["label"].map(1 / counts).to_numpy(),\n                                            len(train), replacement=True)\n        train_loader = make_loader(train, **options, shuffle=True, sampler=sampler)\n        weights = torch.ones(len(LABELS), device=device)\n        if class_weighting != "none":\n            weights = torch.tensor(len(train) / (len(LABELS) * counts.to_numpy()),\n                                   dtype=torch.float32, device=device)\n            if class_weighting == "sqrt":\n                weights = weights.sqrt()\n        if specialist is not None:\n            weights[LABEL_TO_ID[specialist]] *= 2\n        print(f"Training {head_type.upper()} head | encoder LR {ENCODER_LR:g} | head LR {HEAD_LR:g} | "\n              f"schedule {schedule} | seed {model_seed} | specialist {specialist}", flush=True)\n        model, history = train_model(tokenizer, train_loader, val_loader, device,\n                                     nn.CrossEntropyLoss(), weights, epochs=epochs,\n                                     patience=patience, fgsm_epsilon=fgsm_epsilon, head_type=head_type,\n                                     encoder_lr=ENCODER_LR, head_lr=HEAD_LR, schedule=schedule)\n        val = predict_probabilities(model, val_loader, device)\n        if checkpoint is not None:\n            saved = {"model": {name: tensor.cpu() for name, tensor in model.state_dict().items()},\n                     "validation": val, "history": history}\n            with NamedTemporaryFile(dir=checkpoint_dir, suffix=".tmp", delete=False) as file:\n                temporary = Path(file.name)\n            try:\n                torch.save(saved, temporary)\n                temporary.replace(checkpoint)\n            finally:\n                temporary.unlink(missing_ok=True)\n        return (val, predict_probabilities(model, make_loader(test, **options), device),\n                predict_probabilities(model, make_loader(benchmark, **options), device), history)\n\n    if ensemble:\n        members = [(seed, "negative", "cnn"), (seed + 1, "neutral", "mlp"),\n                   (seed + 2, "positive", "mlp")]\n    else:\n        members = [(seed, None, "mlp")]\n    results = [fit(*member) for member in members]\n    histories = {label or "bert": result[3]\n                 for (_, label, _), result in zip(members, results)}\n\n    validation_members = [result[0] for result in results]\n    voting = VotingRule("equal soft")\n    if ensemble:\n        voting = select_vote(validation_members, validation, targets, groups.loc[validation.index],\n                             review_path, sweep_path, tune_neutral)\n    validation_probs, test_probs, benchmark_probs = (\n        vote([result[i] for result in results], voting) for i in range(3)\n    )\n    predicted = validation_probs.argmax(dim=1).tolist()\n    print(classification_report(targets, predicted, labels=list(range(len(LABELS))),\n                                target_names=LABELS, digits=4, zero_division=0))\n    print(confusion_matrix(targets, predicted, labels=list(range(len(LABELS)))))\n    for name, frame, scores in (\n        ("Validation", validation, validation_probs), ("Held-out test", test, test_probs)\n    ):\n        score = f1_score(\n            frame["label"].map(LABEL_TO_ID), scores.argmax(dim=1).tolist(),\n            labels=list(range(len(LABELS))), average="macro", zero_division=0,\n        )\n        print(f"{name} macro-F1 {score:.4f}")\n\n    predictions = benchmark[["id"]].assign(\n        label=[LABELS[i] for i in benchmark_probs.argmax(dim=1).tolist()]\n    )\n    return predictions, histories\n\n\n@dataclass(frozen=True)\nclass VotingRule:\n    name: str\n    weights: tuple = ()\n    neutral_multiplier: float = 1.0\n\n\ndef vote(probabilities, rule):\n    members = torch.stack(probabilities)\n    if rule.weights:\n        weights = members.new_tensor(rule.weights).view(-1, 1, 1)\n        scores = (members * weights).sum(dim=0)\n    else:\n        scores = members.mean(dim=0)\n    if rule.name == "hard":\n        counts = nn.functional.one_hot(members.argmax(dim=2), num_classes=len(LABELS)).sum(dim=0)\n        scores = scores.masked_fill(counts != counts.amax(dim=1, keepdim=True), -torch.inf)\n    scores[:, LABEL_TO_ID["neutral"]] *= rule.neutral_multiplier\n    return scores\n\n\ndef select_vote(probabilities, validation, targets, groups, review_path, sweep_path, tune_neutral):\n    targets = torch.tensor(targets)\n    names = ("negative_cnn", "neutral_mlp", "positive_mlp")\n    individual = [probs.argmax(dim=1) for probs in probabilities]\n    for name, predictions in zip(names, individual):\n        print(name, classification_report(targets, predictions, labels=[0, 1, 2],\n                                          target_names=LABELS, digits=4, zero_division=0))\n    for i, j in ((0, 1), (0, 2), (1, 2)):\n        print(f"{names[i]} vs {names[j]}: {int((individual[i] != individual[j]).sum())}"\n              f"/{len(targets)} validation predictions differ")\n    cnn, neutral, positive = [pred == targets for pred in individual]\n    print(f"CNN alone correct: {int((cnn & ~neutral & ~positive).sum())}; "\n          f"both MLPs correct and CNN wrong: {int((~cnn & neutral & positive).sum())}")\n\n    grid = [VotingRule("grid", (i / 10, j / 10, (10 - i - j) / 10))\n            for i in range(11) for j in range(11 - i)]\n    # Prefer the equal soft control on ties; among grid ties prefer balanced weights.\n    grid.sort(key=lambda rule: sum((w - 1 / 3) ** 2 for w in rule.weights))\n    rules = [VotingRule("equal soft"), VotingRule("hard"), *grid]\n    predictions = {}\n\n    def labels(rule):\n        if rule not in predictions:\n            predictions[rule] = vote(probabilities, rule).argmax(dim=1)\n        return predictions[rule]\n\n    def score(rule, rows):\n        return f1_score(targets[rows], labels(rule)[rows], labels=[0, 1, 2],\n                        average="macro", zero_division=0)\n\n    def select(rows):\n        best = max(rules, key=lambda rule: score(rule, rows))\n        if not tune_neutral:\n            return best\n        # Keep 1.0 first so a tie preserves the unadjusted decision.\n        candidates = [VotingRule(best.name, best.weights, m) for m in (1., .95, 1.05, .90, 1.10)]\n        return max(candidates, key=lambda rule: score(rule, rows))\n\n    folds = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)\n                 .split(validation, targets.numpy(), groups))\n    crossfit = torch.empty_like(targets)\n    for k, (train, held) in enumerate(folds, 1):\n        rule = select(train)\n        crossfit[held] = labels(rule)[held]\n        print(f"Fold {k} selected {rule} | held macro-F1 {score(rule, held):.4f}")\n    all_rows = slice(None)\n    crossfit_f1 = f1_score(targets, crossfit, labels=[0, 1, 2], average="macro", zero_division=0)\n    print(f"Grouped out-of-fold voting selection {crossfit_f1:.4f} | "\n          f"equal soft {score(rules[0], all_rows):.4f} | hard {score(rules[1], all_rows):.4f}")\n    chosen = select(all_rows)\n    selected = labels(chosen)\n    if sweep_path is not None:\n        records = []\n        report_rules = list(rules)\n        if tune_neutral:\n            report_rules.extend(VotingRule(chosen.name, chosen.weights, m) for m in (.90, .95, 1.05, 1.10))\n        for rule in report_rules:\n            predicted = labels(rule)\n            positive = predicted == LABEL_TO_ID["neutral"]\n            actual = targets == LABEL_TO_ID["neutral"]\n            correct = int((positive & actual).sum())\n            weights = rule.weights or ((1/3,) * 3 if rule.name == "equal soft" else (None,) * 3)\n            records.append(dict(rule=rule.name, cnn=weights[0], neutral_mlp=weights[1],\n                                positive_mlp=weights[2], multiplier=rule.neutral_multiplier,\n                                macro_f1=score(rule, all_rows), selected=rule == chosen,\n                                neutral_precision=correct / max(1, int(positive.sum())),\n                                neutral_recall=correct / max(1, int(actual.sum())),\n                                **{f"fold_{k}": score(rule, held)\n                                   for k, (_, held) in enumerate(folds, 1)}))\n        path = Path(sweep_path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        pd.DataFrame(records).to_csv(path, index=False)\n    review = validation[["id", "text", "label"]].copy()\n    for name, pred in zip(names, individual):\n        review[name] = [LABELS[i] for i in pred.tolist()]\n    review["ensemble"] = [LABELS[i] for i in selected.tolist()]\n    errors = review[((targets != selected) & (targets < 2) & (selected < 2)).numpy()]\n    print(f"Negative -> neutral: {int(((targets == 0) & (selected == 1)).sum())}; "\n          f"neutral -> negative: {int(((targets == 1) & (selected == 0)).sum())}")\n    if review_path is not None:\n        path = Path(review_path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        errors.to_csv(path, index=False)\n        print(f"Saved {len(errors)} validation disagreements to {path}")\n    return chosen\n\n\ndef train_model(tokenizer, train_loader, val_loader, device, loss_fn, class_weights,\n                *, epochs, patience, fgsm_epsilon, head_type, encoder_lr, head_lr, schedule):\n    model = SentimentModel(len(tokenizer), head_type=head_type).to(device)\n    optimizer = torch.optim.AdamW(\n        [\n            {"params": model.encoder.parameters(), "lr": encoder_lr},\n            {"params": model.head.parameters(), "lr": head_lr},\n        ],\n        weight_decay=0.01,\n    )\n    total_steps = epochs * len(train_loader)\n    make_scheduler = (get_cosine_schedule_with_warmup if schedule == "cosine"\n                      else get_linear_schedule_with_warmup)\n    scheduler = make_scheduler(\n        optimizer, num_warmup_steps=round(0.1 * total_steps), num_training_steps=total_steps,\n    )\n\n    best_f1 = -1.0\n    best_epoch = 0\n    best_state = None\n    history = []\n    for epoch in range(1, epochs + 1):\n        train_metrics = run_epoch(\n            model, train_loader, loss_fn, device, optimizer, scheduler,\n            class_weights, fgsm_epsilon,\n        )\n        val_metrics = run_epoch(model, val_loader, loss_fn, device)\n        history.append((train_metrics, val_metrics))\n        print(\n            f"Epoch {epoch}/{epochs} | "\n            f"train loss {train_metrics[0]:.4f}, macro-F1 {train_metrics[1]:.4f} | "\n            f"val loss {val_metrics[0]:.4f}, macro-F1 {val_metrics[1]:.4f}",\n            flush=True,\n        )\n        if val_metrics[1] > best_f1:\n            best_f1 = val_metrics[1]\n            best_epoch = epoch\n            best_state = {\n                name: tensor.detach().to(device="cpu", copy=True)\n                for name, tensor in model.state_dict().items()\n            }\n        if epoch - best_epoch >= patience:\n            print(f"Early stopping after epoch {epoch}: validation macro-F1 stalled.")\n            break\n\n    print(f"Best validation macro-F1 {best_f1:.4f} at epoch {best_epoch}.")\n    model.load_state_dict(best_state)\n    return model, history\n\n\nif __name__ == "__main__":\n    main()\n'

In [ ]:
%cd /content/Intro_to_AI_HW/IAI_HW3
from pathlib import Path
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoTokenizer, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
# Load the local baseline snapshot in memory so Colab uses the same revision.
import sys
import types
hw = types.ModuleType("hw3_baseline_snapshot")
hw.__file__ = str(Path("main.py").resolve())
sys.modules[hw.__name__] = hw
exec(compile(BASELINE_SOURCE, hw.__file__, "exec"), hw.__dict__)
from hw3_baseline_snapshot import ROOT, LABELS, LABEL_TO_ID, preprocess, has_content, CNNHead, BASE_MODEL
from transformers import BertModel
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training.")

In [ ]:
seed = 42
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = pd.read_csv(ROOT / "dataset" / "dataset.csv", keep_default_na=False)
if set(data["label"]) != set(LABELS):
    raise ValueError(f"Expected exactly these sentiment labels: {LABELS}")

groups = data["text"].map(lambda text: preprocess(text).casefold())
outer_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
train_ids, held_out_ids = next(outer_split.split(data, data["label"], groups))
train, held_out = data.iloc[train_ids], data.iloc[held_out_ids]
inner_split = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed)
val_ids, test_ids = next(
    inner_split.split(held_out, held_out["label"], groups.iloc[held_out_ids])
)
validation, test = held_out.iloc[val_ids], held_out.iloc[test_ids]
keep = train["text"].map(has_content)
removed = train.loc[~keep, "label"].value_counts().reindex(LABELS, fill_value=0)
train = train.loc[keep]
if set(train["label"]) != set(LABELS):
    raise ValueError("Filtering nearly empty posts left a training class without examples.")
print(f"Removed nearly empty training posts by class: {removed.to_dict()}")
print(f"Device: {device} | train: {len(train)} | validation: {len(validation)} | test: {len(test)}", flush=True)

benchmark = pd.read_csv(ROOT / "dataset" / "benchmark.csv", keep_default_na=False)
targets = validation["label"].map(LABEL_TO_ID).tolist()
counts = train["label"].value_counts().reindex(LABELS)

In [ ]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, head_type="mlp", outputs=3):
        super().__init__()
        self.encoder = BertModel.from_pretrained(BASE_MODEL, add_pooling_layer=False)
        self.encoder.resize_token_embeddings(vocab_size)
        if head_type == "cnn":
            self.head = CNNHead(self.encoder.config.hidden_size)
            if outputs != len(LABELS):
                self.head.classifier = nn.Linear(self.head.classifier.in_features, outputs)
            return
        self.head = nn.Sequential(
            nn.Linear(self.encoder.config.hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, outputs),
        )

    def forward(self, inputs, embeddings=None):
        if embeddings is not None:
            inputs = {name: tensor for name, tensor in inputs.items() if name != "input_ids"}
            inputs["inputs_embeds"] = embeddings
        hidden = self.encoder(**inputs).last_hidden_state
        if isinstance(self.head, CNNHead):
            return self.head(hidden, inputs["attention_mask"])
        return self.head(hidden[:, 0])

In [ ]:
def run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None,
              class_weights=None, fgsm_epsilon=0.0):
    training = optimizer is not None
    model.train(training)
    loss_sum = 0.0
    predictions = []
    targets = []
    with torch.set_grad_enabled(training):
        for inputs, labels in loader:
            inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            labels = labels.to(device)
            if training:
                optimizer.zero_grad()
            embeddings = None
            if training and fgsm_epsilon > 0:
                embeddings = model.encoder.get_input_embeddings()(inputs["input_ids"])
            logits = model(inputs) if embeddings is None else model(inputs, embeddings)
            loss = loss_fn(logits, labels)
            if training:
                objective = loss
                if class_weights is not None:
                    objective = nn.functional.cross_entropy(logits, labels, weight=class_weights)
                if embeddings is not None:
                    gradient, = torch.autograd.grad(objective, embeddings, retain_graph=True)
                    perturbation = fgsm_epsilon * gradient.sign()
                    perturbation = perturbation * inputs["attention_mask"].unsqueeze(-1)
                    adversarial_logits = model(inputs, embeddings + perturbation)
                    adversarial_loss = nn.functional.cross_entropy(
                        adversarial_logits, labels, weight=class_weights,
                    )
                    objective = (objective + adversarial_loss) / 2
                objective.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            loss_sum += loss.item() * labels.size(0)
            predictions.extend(logits.detach().argmax(dim=1).tolist())
            targets.extend(labels.tolist())

    macro_f1 = f1_score(
        targets, predictions, labels=list(range(logits.size(1))),
        average="macro", zero_division=0,
    )
    return loss_sum / len(targets), macro_f1

def train_model(tokenizer, train_loader, val_loader, device, loss_fn, class_weights,
                *, epochs, patience, fgsm_epsilon, head_type, encoder_lr, head_lr, schedule, outputs):
    model = SentimentModel(len(tokenizer), head_type=head_type, outputs=outputs).to(device)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": encoder_lr},
            {"params": model.head.parameters(), "lr": head_lr},
        ],
        weight_decay=0.01,
    )
    total_steps = epochs * len(train_loader)
    make_scheduler = (get_cosine_schedule_with_warmup if schedule == "cosine"
                      else get_linear_schedule_with_warmup)
    scheduler = make_scheduler(
        optimizer, num_warmup_steps=round(0.1 * total_steps), num_training_steps=total_steps,
    )

    best_f1 = -1.0
    best_epoch = 0
    best_state = None
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(
            model, train_loader, loss_fn, device, optimizer, scheduler,
            class_weights, fgsm_epsilon,
        )
        val_metrics = run_epoch(model, val_loader, loss_fn, device)
        history.append((train_metrics, val_metrics))
        print(
            f"Epoch {epoch}/{epochs} | "
            f"train loss {train_metrics[0]:.4f}, macro-F1 {train_metrics[1]:.4f} | "
            f"val loss {val_metrics[0]:.4f}, macro-F1 {val_metrics[1]:.4f}",
            flush=True,
        )
        if val_metrics[1] > best_f1:
            best_f1 = val_metrics[1]
            best_epoch = epoch
            best_state = {
                name: tensor.detach().to(device="cpu", copy=True)
                for name, tensor in model.state_dict().items()
            }
        if epoch - best_epoch >= patience:
            print(f"Early stopping after epoch {epoch}: validation macro-F1 stalled.")
            break

    print(f"Best validation macro-F1 {best_f1:.4f} at epoch {best_epoch}.")
    model.load_state_dict(best_state)
    return model, history

In [ ]:
def train_member(index, binary):
    label = LABELS[index]
    head = "cnn" if label == "negative" else "mlp"
    torch.manual_seed(42 + index)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.add_special_tokens({"additional_special_tokens": [hw.COMPANY_TOKEN, hw.GMAIL_TOKEN]})
    def loader(frame, shuffle=False):
        batches = hw.make_loader(frame, tokenizer, 16, 128, shuffle=shuffle)
        if binary:
            # Keep tokenization, batching and ordering from the original loader.
            collate = batches.collate_fn
            def binary_collate(batch):
                inputs, labels = collate(batch)
                return inputs, labels.eq(index).long()
            batches.collate_fn = binary_collate
        return batches

    frequencies = counts.to_list()
    target = index
    if binary:
        frequencies = [len(train) - frequencies[index], frequencies[index]]
        target = 1
    weights = (len(train) / (len(frequencies) * torch.tensor(frequencies, device=device))).sqrt()
    weights[target] *= 2
    model, _ = train_model(tokenizer, loader(train, True), loader(validation), device,
        nn.CrossEntropyLoss(), weights, epochs=5, patience=2, fgsm_epsilon=0,
        head_type=head, encoder_lr=hw.ENCODER_LR, head_lr=hw.HEAD_LR,
        schedule="linear", outputs=len(frequencies))
    return [hw.predict_probabilities(model, loader(frame), device)
            for frame in (validation, test, benchmark)]

In [ ]:
baseline = [train_member(i, False) for i in range(3)]
rule = hw.select_vote([member[0] for member in baseline], validation, targets,
                      groups.loc[validation.index], None, None, True)
baseline_test = hw.vote([member[1] for member in baseline], rule).argmax(1)
binary = [train_member(i, True) for i in range(3)]
def yes_scores(split):
    return torch.stack([member[split][:, 1] for member in binary], dim=1)
binary_test = yes_scores(1).argmax(1)

In [ ]:
truth = test["label"].map(LABEL_TO_ID)
rows = []
for name, predicted in [("Original ensemble", baseline_test), ("Binary specialists", binary_test)]:
    print(name)
    print(classification_report(truth, predicted, labels=[0, 1, 2], target_names=LABELS, zero_division=0))
    rows.append((name, accuracy_score(truth, predicted), f1_score(truth, predicted, average="macro")))
summary = pd.DataFrame(rows, columns=["Model", "Held-out accuracy", "Held-out macro-F1"]).set_index("Model")
display(summary, (summary.iloc[1] - summary.iloc[0]).rename("Binary minus original"))
scores = yes_scores(2)
result = benchmark[["id"]].copy()
for i, label in enumerate(LABELS):
    result[label + "_p_yes"] = scores[:, i].numpy()
    result[label + "_answer"] = ["yes" if p > 0.5 else "no" for p in scores[:, i]]
result["label"] = [LABELS[i] for i in scores.argmax(1)]
result.to_csv("/content/binary_specialist_predictions.csv", index=False)
display(result.head())